# [3/4] TRANSFORMER BRANCH – RT-DETR-L (detection) + ViT-B/16 & ViT-S/16 (classification)
### Run AFTER notebook 02 (self-contained; same dataset and split, seed 42)

| Model | Task | HPO |
|---|---|---|
| **RT-DETR-L** (CVPR 2024, Ultralytics) | Detection, 10 classes | lr (2 configs) → best × 40 epochs |
| **ViT-B/16** (timm, ImageNet) | Crop classification, 10 classes | lr × weight decay (4 configs) → best × 6 epochs |
| **ViT-S/16** | Crop classification, 10 classes | runs the winning ViT-B config (model-size comparison) |

Kaggle settings: **GPU T4 x2** + **Internet ON** (required — pip installs ultralytics/timm and downloads pretrained RT-DETR/ViT weights) + **+ Add Input dataset `ppe-unified-v1`**. Estimated ~8–9 h.

## Dataset: PPE_Unified_v1 (shared by all three notebooks — KEEP THIS CELL IDENTICAL IN ALL THREE FILES)
**Safety-PPE Unified v1** — a dataset built by merging three public datasets (CC BY 4.0): Hard Hat v2 (computer-vision-filqz) + PPE v1 (testcasque) + PPE v5 (mohamed-traore-2ekkp). **12 classes = 6 compliant/violation pairs**, ~18k images that have been deduplicated by source image, cross-deduplicated by MD5, **normalised to 640×640 letterbox**, and pre-split **70% train / 20% val / 10% test grouped by source image (leakage-free)**. Details in the dataset's `DATASET_CARD.md`.

**How to load:** the dataset is on Kaggle (`ppe-unified-v1`); in EACH notebook click **+ Add Input → Your Datasets → ppe-unified-v1**. No API key needed — data is read offline from Input. (Note: Internet ON is still required in notebooks 01/03 to install libraries and download pretrained weights.)

The split is fixed inside the dataset, so all three notebooks are compared on exactly the same partition. Class-imbalance handling during training: (1) repeat-factor oversampling of rare classes (train only); (2) crop undersampling to MAX_CROPS_PER_CLASS; (3) class weights in the loss; (4) model selection by macro-F1 / per-class mAP (classes with <10 test samples — e.g. no_boots — are excluded from worst_class_f1 and reported separately). Rule: every hyperparameter is chosen on VAL; TEST is touched exactly once, at the end.

In [ ]:
# ====================== CONFIG (Transformer) ======================
BRANCH     = "Transformer"
QUICK_TEST = False    # True: fast pipeline smoke test (~15-20 min)
RUN_HPO    = True

SEED = 42
SPLIT = (0.7, 0.2, 0.1)
RFS_T, REPEAT_CAP = 0.15, 4                 # repeat-factor oversampling of rare classes (train only)
CROP_SIZE, CROP_SIZE_VIT = 128, 224
MAX_CROPS_PER_CLASS = 200 if QUICK_TEST else 2500

import os, random, time, shutil, json, numpy as np
random.seed(SEED); np.random.seed(SEED)
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/results", exist_ok=True)
HPO_TABLES = {}

# ============ RESUME: path to the previous output of THIS notebook (if any) ============
RESUME_FROM = ""
if RESUME_FROM and os.path.exists(RESUME_FROM):
    for sub in ["state.json", "results", "runs"]:
        src = os.path.join(RESUME_FROM, sub)
        if os.path.isdir(src): shutil.copytree(src, os.path.join(WORK, sub), dirs_exist_ok=True)
        elif os.path.isfile(src): shutil.copy(src, os.path.join(WORK, sub))
    print("CHECKPOINT LOADED FROM:", RESUME_FROM)

STATE_PATH = f"{WORK}/state.json"
STATE = json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)
def done(key):    return key in STATE
def mark(key, value=True): STATE[key] = value; save_state()
def free():
    import gc, torch; gc.collect(); torch.cuda.empty_cache()

print(f"[{BRANCH}] QUICK_TEST={QUICK_TEST} | RUN_HPO={RUN_HPO} | stages done:", [k for k in STATE] or "(none yet)")

In [ ]:
%pip install -q -U ultralytics timm
import torch
assert torch.cuda.is_available(), "GPU not enabled! Settings -> Accelerator -> GPU T4 x2"
cap = torch.cuda.get_device_capability(0); gpu_name = torch.cuda.get_device_name(0)
if cap[0] < 7:
    raise SystemError(f"GPU {gpu_name} (sm_{cap[0]}{cap[1]}) is not supported by PyTorch. Switch to GPU T4 x2!")
print("GPU:", gpu_name, "- OK")

In [ ]:
from pathlib import Path
from collections import Counter
import yaml, glob

# ==== Locate PPE_Unified_v1 under /kaggle/input (must be Added as Input) ====
DS_ROOT = None
for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
    root = os.path.dirname(c)
    if os.path.isdir(f"{root}/train/images") and os.path.isdir(f"{root}/val/images") and os.path.isdir(f"{root}/test/images"):
        DS_ROOT = root; break
assert DS_ROOT, ("PPE_Unified_v1 not found! Upload the PPE_Unified_v1 folder to Kaggle "
                 "(Create -> New Dataset) then '+ Add Input' it into this notebook.")
print("Dataset input:", DS_ROOT)

raw_yaml = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))
raw_names = raw_yaml["names"]
if isinstance(raw_names, dict): raw_names = [raw_names[k] for k in sorted(raw_names)]
CLASSES = list(raw_names); NUM_CLS = len(CLASSES)
print(f"{NUM_CLS} classes:", CLASSES)

# ==== Copy to /kaggle/working (input is read-only; YOLO needs to write labels.cache) ====
# The split is ALREADY fixed in the dataset (grouped by source image, leakage-free) -> do NOT re-split.
DATA = f"{WORK}/dataset"
CAPS = {"train": 300, "val": 80, "test": 40} if QUICK_TEST else {}
splits = {}
for sp in ["train", "val", "test"]:
    os.makedirs(f"{DATA}/{sp}/images", exist_ok=True); os.makedirs(f"{DATA}/{sp}/labels", exist_ok=True)
    files = sorted(os.listdir(f"{DS_ROOT}/{sp}/images"))
    random.Random(SEED).shuffle(files)
    if sp in CAPS: files = files[:CAPS[sp]]
    done_copy = len(os.listdir(f"{DATA}/{sp}/images")) >= len(files)   # resume: skip if already copied
    for fn in files:
        if done_copy: break
        shutil.copy(f"{DS_ROOT}/{sp}/images/{fn}", f"{DATA}/{sp}/images/{fn}")
        lp = f"{DS_ROOT}/{sp}/labels/{Path(fn).stem}.txt"
        if os.path.exists(lp): shutil.copy(lp, f"{DATA}/{sp}/labels/{Path(fn).stem}.txt")
    splits[sp] = files
print("Image count:", {sp: len(v) for sp, v in splits.items()})

# ==== Check image-size consistency (the dataset was letterboxed to 640x640 at build time) ====
from PIL import Image
sample = [f"{DATA}/train/images/{f}" for f in splits["train"][:200]]
dims = Counter(Image.open(p).size for p in sample)
print("Image sizes (200-image sample):", dict(dims.most_common(3)))
assert dims.most_common(1)[0][0] == (640, 640), "Images are not uniformly 640x640 - rebuild the dataset with NORMALIZE_SIZE=640!"

# ==== Class statistics + repeat-factor oversampling (LVIS) - TRAIN split ONLY ====
cls_count, img_classes = Counter(), {}
for sp in ["train", "val", "test"]:
    for lf in os.listdir(f"{DATA}/{sp}/labels"):
        if "_rfs" in lf: continue
        present = Counter()
        for line in open(f"{DATA}/{sp}/labels/{lf}"):
            if not line.strip(): continue
            c = int(line.split()[0]); present[c] += 1; cls_count[(sp, CLASSES[c])] += 1
        if sp == "train": img_classes[lf] = present

from math import sqrt
n_train = len(splits["train"])
f_c = {c: sum(1 for s in img_classes.values() if c in s)/max(1, n_train) for c in range(NUM_CLS)}
r_c = {c: max(1.0, sqrt(RFS_T/max(f_c[c], 1e-6))) for c in range(NUM_CLS)}
print("Repeat factor, rare classes:", {CLASSES[c]: round(r_c[c],1) for c in range(NUM_CLS) if r_c[c] > 1.5})
for c in range(NUM_CLS):
    if cls_count[("train", CLASSES[c])]: cls_count[("train_after_rfs", CLASSES[c])] = cls_count[("train", CLASSES[c])]
added = 0
for lf, present in list(img_classes.items()):
    if not present: continue
    r = min(REPEAT_CAP, max(r_c[c] for c in present))
    if r < 1.5: continue
    stem = Path(lf).stem
    imgs = glob.glob(f"{DATA}/train/images/{stem}.*")
    if not imgs: continue
    ext = Path(imgs[0]).suffix
    for k in range(int(round(r)) - 1):
        dst = f"{DATA}/train/images/{stem}_rfs{k}{ext}"
        if not os.path.exists(dst):
            shutil.copy(imgs[0], dst)
            shutil.copy(f"{DATA}/train/labels/{lf}", f"{DATA}/train/labels/{stem}_rfs{k}.txt")
        for c, cnt in present.items(): cls_count[("train_after_rfs", CLASSES[c])] += cnt
        added += 1
print(f"Oversampling: +{added} duplicated images containing rare classes (train {n_train} -> {n_train+added})")

names_yaml = "\n".join(f"  {i}: {nm}" for i, nm in enumerate(CLASSES))
with open(f"{DATA}/data.yaml", "w") as f:
    f.write(f"path: {DATA}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n{names_yaml}\n")

import pandas as pd
df_dist = pd.DataFrame([{"split":k[0],"class":k[1],"boxes":v} for k,v in cls_count.items()]).pivot_table(index="class",columns="split",values="boxes",fill_value=0)
df_dist.to_csv(f"{WORK}/results/dataset_distribution.csv"); df_dist

In [ ]:
import cv2
from PIL import Image

def build_crops(split, size):
    X, y = [], []; per_class = Counter()
    files = sorted(os.listdir(f"{DATA}/{split}/labels"))
    random.Random(SEED).shuffle(files)
    for lf in files:
        stem = Path(lf).stem
        if "_rfs" in stem: continue
        for ext in [".png",".jpg",".jpeg"]:
            ip = f"{DATA}/{split}/images/{stem}{ext}"
            if os.path.exists(ip): break
        else: continue
        img = Image.open(ip).convert("RGB"); W,H = img.size
        for line in open(f"{DATA}/{split}/labels/{lf}"):
            if not line.strip(): continue
            c,xc,yc,bw,bh = map(float, line.split()); c=int(c)
            if per_class[c] >= MAX_CROPS_PER_CLASS: continue
            w,h = bw*W, bh*H
            if w<16 or h<16: continue
            pad=0.15
            box = (max(0,(xc-bw/2)*W-pad*w), max(0,(yc-bh/2)*H-pad*h),
                   min(W,(xc+bw/2)*W+pad*w), min(H,(yc+bh/2)*H+pad*h))
            X.append(np.asarray(img.crop(box).resize((size,size)), dtype=np.uint8)); y.append(c)
            per_class[c]+=1
    return np.array(X), np.array(y)

crops = {sp: build_crops(sp, CROP_SIZE) for sp in ["train","val","test"]}
for sp in crops:
    print(sp, crops[sp][0].shape, {CLASSES[i]: int(v) for i,v in enumerate(np.bincount(crops[sp][1], minlength=NUM_CLS))})
(Xtr,ytr),(Xva,yva),(Xte,yte) = crops["train"], crops["val"], crops["test"]

from sklearn.utils.class_weight import compute_class_weight
present_cls = np.unique(ytr)
w = compute_class_weight("balanced", classes=present_cls, y=ytr)
CLASS_WEIGHT = {int(c): float(v) for c, v in zip(present_cls, w)}
CW_TENSOR = np.ones(NUM_CLS, dtype="float32")
for c, v in CLASS_WEIGHT.items(): CW_TENSOR[c] = v
print("Class weights:", {CLASSES[c]: round(v,2) for c,v in CLASS_WEIGHT.items()})

from sklearn.metrics import accuracy_score, f1_score, classification_report
def multiclass_metrics(y_true, pred):
    acc = accuracy_score(y_true, pred)
    macro = f1_score(y_true, pred, average="macro")
    per_cls = f1_score(y_true, pred, average=None, labels=list(range(NUM_CLS)), zero_division=0)
    support = np.bincount(y_true, minlength=NUM_CLS)
    valid = [c for c in range(NUM_CLS) if support[c] >= 10]   # classes with <10 samples (e.g. no_boots) excluded from worst - reported separately
    worst = float(np.min([per_cls[c] for c in valid])) if valid else 0.0
    return acc, macro, worst

## 1. RT-DETR-L – HPO (2 lr configs, AdamW) → final training, 40 epochs

In [ ]:
from ultralytics import RTDETR
IMG_FINAL, IMG_SCREEN = 640, 512
EP_SCREEN = 2 if QUICK_TEST else 5
EP_FINAL  = 3 if QUICK_TEST else 30

RT_GRID = [{"lr0": 1e-4}, {"lr0": 5e-5}]
rows=[]; best_cfg_rt = RT_GRID[0]
if RUN_HPO:
    for i, cfg in enumerate(RT_GRID):
        key = f"rtdetr_hpo{i}"
        if done(key): rows.append(STATE[key]); print("skip:", STATE[key]); continue
        m = RTDETR("rtdetr-l.pt"); t0=time.time()
        m.train(data=f"{DATA}/data.yaml", epochs=EP_SCREEN, imgsz=IMG_SCREEN, batch=8,
                seed=SEED, project=f"{WORK}/runs", name=f"rtdetr_hpo{i}", plots=False, verbose=False,
                exist_ok=True, optimizer="AdamW", **cfg)
        v = m.val(data=f"{DATA}/data.yaml", split="val", verbose=False)
        row = {**cfg, "mAP50_val": round(v.box.map50,4), "min": round((time.time()-t0)/60,1)}
        rows.append(row); mark(key, row); print(row); del m; free()
    hpo_rt = pd.DataFrame(rows).sort_values("mAP50_val", ascending=False)
    HPO_TABLES["RT-DETR"] = hpo_rt; display(hpo_rt)
    best_cfg_rt = {"lr0": float(hpo_rt.iloc[0]["lr0"])}

res_det = {}
if done("rtdetr_final"):
    res_det["RT-DETR-L (Transformer)"] = STATE["rtdetr_final"]; print("skip")
else:
    last_ckpt = f"{WORK}/runs/rtdetr_final/weights/last.pt"; t0=time.time()
    if os.path.exists(last_ckpt):
        print("RESUMING from", last_ckpt); rt = RTDETR(last_ckpt); rt.train(resume=True)
    else:
        rt = RTDETR("rtdetr-l.pt")
        rt.train(data=f"{DATA}/data.yaml", epochs=EP_FINAL, imgsz=IMG_FINAL, batch=8,
                 seed=SEED, project=f"{WORK}/runs", name="rtdetr_final", patience=10, plots=True,
                 exist_ok=True, optimizer="AdamW", lr0=best_cfg_rt["lr0"])
    mv = rt.val(data=f"{DATA}/data.yaml", split="val")
    mt = rt.val(data=f"{DATA}/data.yaml", split="test")
    res_det["RT-DETR-L (Transformer)"] = {"cfg": str(best_cfg_rt),
        "mAP50_val": round(mv.box.map50,4), "mAP50_test": round(mt.box.map50,4),
        "mAP50_95_test": round(mt.box.map,4), "precision_test": round(mt.box.mp,4),
        "recall_test": round(mt.box.mr,4), "infer_ms": round(mt.speed["inference"],2),
        "train_min": round((time.time()-t0)/60,1)}
    print("Per-class mAP50:", dict(zip([CLASSES[i] for i in mt.box.ap_class_index], np.round(mt.box.ap50,4))))
    mark("rtdetr_final", res_det["RT-DETR-L (Transformer)"]); del rt; free()
res_det["RT-DETR-L (Transformer)"]

## 2. ViT-B/16 – HPO (lr × weight decay) → final training; plus ViT-S/16 for a model-size comparison

In [ ]:
import timm, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
device = "cuda"
EP_SCREEN_VIT = 1 if QUICK_TEST else 2
EP_FINAL_VIT  = 1 if QUICK_TEST else 6
MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1); STD = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

class CropDS(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        img = cv2.resize(self.X[i], (CROP_SIZE_VIT, CROP_SIZE_VIT))
        t = torch.from_numpy(img).permute(2,0,1).float()/255.
        return (t-MEAN)/STD, int(self.y[i])

def train_vit(arch, lr, wd, epochs):
    vit = timm.create_model(arch, pretrained=True, num_classes=NUM_CLS).to(device)
    opt = torch.optim.AdamW(vit.parameters(), lr=lr, weight_decay=wd)
    scaler = torch.cuda.amp.GradScaler()
    crit = nn.CrossEntropyLoss(weight=torch.tensor(CW_TENSOR).to(device))
    dl = DataLoader(CropDS(Xtr, ytr), batch_size=32, shuffle=True, num_workers=2)
    for ep in range(epochs):
        vit.train()
        for xb,yb in dl:
            xb,yb = xb.to(device), yb.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(): loss = crit(vit(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    return vit

@torch.no_grad()
def vit_pred(vit, X, y):
    vit.eval(); out=[]
    for xb,_ in DataLoader(CropDS(X, y), batch_size=64, num_workers=2):
        out.append(vit(xb.to(device)).argmax(1).cpu())
    return torch.cat(out).numpy()

VIT_GRID = [{"lr":5e-5,"wd":0.01},{"lr":5e-5,"wd":0.1},{"lr":1e-5,"wd":0.01},{"lr":1e-5,"wd":0.1}]
rows=[]; best_cfg_vit = VIT_GRID[0]
if RUN_HPO:
    for i, cfg in enumerate(VIT_GRID):
        key = f"vit_hpo{i}"
        if done(key): rows.append(STATE[key]); print("skip:", STATE[key]); continue
        t0=time.time(); vit = train_vit("vit_base_patch16_224", cfg["lr"], cfg["wd"], EP_SCREEN_VIT)
        pred = vit_pred(vit, Xva, yva)
        acc, macro, worst = multiclass_metrics(yva, pred)
        row = {**cfg, "macro_f1_val": round(macro,4), "min": round((time.time()-t0)/60,1)}
        rows.append(row); mark(key, row); print(row); del vit; free()
    hpo_vit = pd.DataFrame(rows).sort_values("macro_f1_val", ascending=False)
    HPO_TABLES["ViT"] = hpo_vit; display(hpo_vit)
    best_cfg_vit = {"lr": float(hpo_vit.iloc[0]["lr"]), "wd": float(hpo_vit.iloc[0]["wd"])}

res_cls = {}
for arch, label in [("vit_base_patch16_224","ViT-B/16 (Transformer)"), ("vit_small_patch16_224","ViT-S/16 (Transformer)")]:
    key = f"{arch}_final"
    if done(key): res_cls[label] = STATE[key]; print("skip:", STATE[key]); continue
    t0=time.time(); vit = train_vit(arch, best_cfg_vit["lr"], best_cfg_vit["wd"], EP_FINAL_VIT)
    pred = vit_pred(vit, Xte, yte)
    acc, macro, worst = multiclass_metrics(yte, pred)
    res_cls[label] = {"cfg": f"{arch}, "+str(best_cfg_vit),
        "accuracy":round(acc,4), "macro_f1":round(macro,4), "worst_class_f1":round(worst,4),
        "train_min":round((time.time()-t0)/60,1)}
    print(res_cls[label])
    print(classification_report(yte, pred, labels=list(range(NUM_CLS)), target_names=CLASSES, zero_division=0))
    torch.save(vit.state_dict(), f"{WORK}/results/{arch}_best.pt")
    mark(key, res_cls[label]); del vit; free()

## 3. Export normalised results

In [ ]:
# ====== EXPORT normalised summary.csv for notebook 04 to aggregate ======
rows = []
for name, d in res_det.items():
    rows.append({"branch": BRANCH, "task": "detection", "model": name, **d})
for name, d in res_cls.items():
    rows.append({"branch": BRANCH, "task": "classification", "model": name,
                 **{k: v for k, v in d.items() if k != "_name"}})
df_sum = pd.DataFrame(rows)
df_sum.to_csv(f"{WORK}/results/summary.csv", index=False)
for name, tbl in HPO_TABLES.items():
    tbl.to_csv(f"{WORK}/results/hpo_{name.replace('/','').replace(' ','_')}.csv", index=False)
print("Saved results/summary.csv and the HPO tables. After COMMIT, use this output as Input for notebook 04.")
display(df_sum)

## Checkpoint & resume (if this notebook is interrupted before finishing)
The notebook saves progress to `state.json` + `runs/` after every config/model. If the session is cut short: (1) it runs via **Save & Run All (Commit)**, so the output is preserved in that version; (2) open the notebook → **+ Add Input → Your Work** → select the version that stopped; (3) set `RESUME_FROM = "/kaggle/input/<slug>"` in the Config cell; (4) Run All again — finished stages are skipped and detection training continues from `last.pt`.